In [ ]:
from langgraph.graph import StateGraph,START,END
from pydantic import BaseModel
from langgraph.types import interrupt,Command
from langgraph.checkpoint.memory import MemorySaver

In [ ]:
class NodeState(BaseModel):
    name:str
    status:str

In [ ]:
def worker_graph(state:NodeState):
    return {"status": "FAILED"}


In [ ]:
checkpointer=MemorySaver()
def HITL(state:NodeState):
    user_input=interrupt("Please provide input for HITL: ")
    return {"status":  user_input}

In [ ]:
graph = StateGraph(NodeState)
graph.add_node("worker", worker_graph)
graph.add_node("HITL", HITL)
graph.add_edge(START, "worker")
graph.add_edge("worker", "HITL")
graph.add_edge("HITL", END)
config = {
    "configurable": {
        "thread_id": "thread_1"
    }
}

app=graph.compile(
    checkpointer=checkpointer

)

In [ ]:
result = app.invoke(
    {
        "name": "AI Agent",
        "status": "PENDING"
    },
    config=config
)

print(result)

In [8]:
from langgraph.graph import StateGraph, START, END
from pydantic import BaseModel
from langgraph.types import interrupt, Command
from langgraph.checkpoint.memory import MemorySaver


class NodeState(BaseModel):
    name: str
    status: str


def worker_graph(state: NodeState):
    return {
        "status": "FAILED"
    }


def HITL(state: NodeState):

    user_input = interrupt(
        "APPROVED, RETRY, REJECTED"
    )

    if user_input == "APPROVED":
        return {
            "status": "SUCCESS"
        }

    elif user_input == "RETRY":
        return {
            "status": "RETRY"
        }

    else:
        return {
            "status": "REJECTED"
        }


# =========================================
# Conditional Router
# =========================================

def route_after_hitl(state: NodeState):

    if state.status == "RETRY":
        return "HITL"

    return "END"


# =========================================
# Checkpointer
# =========================================

checkpointer = MemorySaver()


# =========================================
# Graph
# =========================================

graph = StateGraph(NodeState)

graph.add_node("worker", worker_graph)
graph.add_node("HITL", HITL)


graph.add_edge(
    START,
    "worker"
)

graph.add_edge(
    "worker",
    "HITL"
)


graph.add_conditional_edges(
    "HITL",
    route_after_hitl,
    {
        "HITL": "HITL",
        "END": END
    }
)


app = graph.compile(
    checkpointer=checkpointer
)

# =========================================
# Thread
# =========================================

config = {
    "configurable": {
        "thread_id": "thread_1"
    }
}


# =========================================
# FIRST EXECUTION
# =========================================
before_resume=app.get_state(config)
result_1 = app.invoke(
    {
        "name": "AI Agent",
        "status": "PENDING"
    },
    config=config
)
after_interrupt=app.get_state(config)
print("FIRST RESULT:")
print(result_1)
print("STATE BEFORE RESUMING")
print(before_resume)
print("STATE AFTER INTERRUPT:")
print(after_interrupt)


# =========================================
# HUMAN DECISION 1
# =========================================

result_2 = app.invoke(
    Command(resume="RETRY"),
    config=config
)
after_resume=app.get_state(config)
print("STATE AFTER RESUMING")
print(after_resume)
print("AFTER RETRY:")
print(result_2)



# =========================================
# HUMAN DECISION 2
# =========================================

result_3 = app.invoke(
    Command(resume="REJECTED"),
    config=config
)

print("FINAL RESULT:")
print(result_3)

FIRST RESULT:
{'name': 'AI Agent', 'status': 'FAILED', '__interrupt__': [Interrupt(value='APPROVED, RETRY, REJECTED', id='483dfd602dee550e52af65dfb418130b')]}
STATE BEFORE RESUMING
StateSnapshot(values={}, next=(), config={'configurable': {'thread_id': 'thread_1'}}, metadata=None, created_at=None, parent_config=None, tasks=(), interrupts=())
STATE AFTER INTERRUPT:
StateSnapshot(values={'name': 'AI Agent', 'status': 'FAILED'}, next=('HITL',), config={'configurable': {'thread_id': 'thread_1', 'checkpoint_ns': '', 'checkpoint_id': '1f1aaa9b-648d-6003-8001-fba81da4ba9e'}}, metadata={'source': 'loop', 'step': 1, 'parents': {}}, created_at='2026-09-07T10:48:50.961582+00:00', parent_config={'configurable': {'thread_id': 'thread_1', 'checkpoint_ns': '', 'checkpoint_id': '1f1aaa9b-6439-63fe-8000-1690fe1aea3d'}}, tasks=(PregelTask(id='dea812ef-6a02-97ee-e1be-0c19b4a8d575', name='HITL', path=('__pregel_pull', 'HITL'), error=None, interrupts=(Interrupt(value='APPROVED, RETRY, REJECTED', id='483dfd